# Part 2: Streaming application using Spark Structured Streaming  
In this part, I implemented Spark Structured Streaming to consume the data from previous kafka producer and perform a prediction.    
Important: 
- This part uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.
- I use my pipeline model from previous model training to do the predictions and persist the results.
- Note for the prediction related to event time: in a real scenario, I use accident_ts as the event time.

## 2.1 Spark Session Configuration

This section creates the Spark environment required for Structured Streaming. The session runs locally with four cores, uses the `Europe/London` timezone so the accident timestamps are interpreted consistently with the UK dataset, and sets a checkpoint base directory for streaming fault tolerance. The Kafka connector package is included in my SparkSession configuration using spark.jars.packages. It is needed because Task 2 uses Spark Structured Streaming to read from Kafka topics and later write the processed Parquet stream outputs back to Kafka topics using .format("kafka")

In [67]:
# Task 2.1 Spark Session Configuration

from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, ArrayType
)

spark_conf = (
    SparkConf()
    .setMaster("local[4]")
    .setAppName("Assignment 2B - Traffic Accident Streaming Prediction")
    .set("spark.sql.session.timeZone", "Europe/London")
    .set("spark.sql.streaming.checkpointLocation", "A2B/checkpoint")
    .set("spark.sql.shuffle.partitions", "4")
    .set(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    )
)

spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("SparkSession created successfully.")
print("Spark version:", spark.version)
print("Session timezone:", spark.conf.get("spark.sql.session.timeZone"))

SparkSession created successfully.
Spark version: 4.1.1
Session timezone: Europe/London


## 2.2 Schema Definition and Static Dataset Loading

In this part, two data sources are used. The collision records are treated as streaming data. Although the column names are read from `streaming_collision.csv` to define the schema, the collision records themselves are not loaded directly as a static Spark DataFrame in this step. Instead, they are sent by the Kafka producer and later consumed as streaming data from the Kafka topic.

The vehicle dataset is treated as static reference data. It is loaded directly from `vehicle.csv` into `vehicle_df` because vehicle information can be considered relatively stable and available from a registration database. This static vehicle DataFrame can later be joined with the streaming collision records during the streaming prediction process.

In [68]:
# Task 2.2 Schema Definition and Static Dataset Loading

import csv

collision_path = "streaming_collision.csv"
vehicle_path = "vehicle.csv"


def get_csv_header(file_path):
    with open(file_path, "r", newline="", encoding="utf-8") as file:
        reader = csv.reader(file)
        header = next(reader)

    return header


collision_columns = get_csv_header(collision_path)
vehicle_columns = get_csv_header(vehicle_path)

print("Collision columns:", len(collision_columns))
print(collision_columns)

print("\nVehicle columns:", len(vehicle_columns))
print(vehicle_columns)


# Streaming collision records arrive as strings, except accident_ts.
collision_schema = StructType(
    [StructField(col_name, StringType(), True) for col_name in collision_columns]
    + [StructField("accident_ts", LongType(), True)]
)

# The producer sends one Kafka message as a list of accident records.
collision_batch_schema = ArrayType(collision_schema)

# Static vehicle data is loaded from CSV.
vehicle_schema = StructType(
    [StructField(col_name, StringType(), True) for col_name in vehicle_columns]
)

vehicle_df = (
    spark.read
    .option("header", "true")
    .schema(vehicle_schema)
    .csv(vehicle_path)
)

vehicle_df.printSchema()
vehicle_df.show(5, truncate=False)

Collision columns: 16
['collision_index', 'longitude', 'latitude', 'date', 'time', 'road_type', 'speed_limit', 'junction_detail', 'junction_control', 'pedestrian_crossing', 'light_conditions', 'weather_conditions', 'road_surface_conditions', 'carriageway_hazards', 'urban_or_rural_area', 'area']

Vehicle columns: 13
['collision_index', 'vehicle_reference', 'vehicle_type', 'vehicle_manoeuvre', 'junction_location', 'skidding_and_overturning', 'hit_object_in_carriageway', 'first_point_of_impact', 'sex_of_driver', 'age_of_driver', 'engine_capacity_cc', 'propulsion_code', 'age_of_vehicle']
root
 |-- collision_index: string (nullable = true)
 |-- vehicle_reference: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- vehicle_manoeuvre: string (nullable = true)
 |-- junction_location: string (nullable = true)
 |-- skidding_and_overturning: string (nullable = true)
 |-- hit_object_in_carriageway: string (nullable = true)
 |-- first_point_of_impact: string (nullable = true)


## 2.3 Kafka Stream Ingestion and Type Conversion

This step reads the streaming accident data that was published by the Kafka producer in Task 1. The producer sends accident records to the Kafka topic a2b_accident_stream, and Spark Structured Streaming subscribes to the same topic through the Kafka broker at kafka:9092.

Kafka stores the message content in the value column. Since Kafka values are received in binary format, the value is first cast into a string. The string is then deserialised from JSON using the previously defined collision_batch_schema. Because the producer sends one Kafka message as a batch of 50 to 100 accident records, the parsed result is an array of records. Therefore, explode() is used to convert each record in the batch into a separate row in the streaming DataFrame.

The accident_ts column is added by the producer as a numeric Unix timestamp to represent the simulated streaming event time. In this step, it is converted into accident_ts_time, a proper Spark timestamp column, so it can be used later for event-time processing, watermarking, and window-based aggregation.

In [69]:
# Task 2.3 Kafka Stream Ingestion and Type Conversion

# Kafka broker and topic configuration.
hostip = "kafka"
raw_topic = "a2b_accident_stream"

# Read streaming messages from the Kafka topic.
# Spark Structured Streaming continuously listens to the Kafka broker.
kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("subscribe", raw_topic)
    .option("startingOffsets", "earliest")
    .load()
)

# Kafka stores message content in the value column as binary.
# The value column is cast to string so that it can be parsed as JSON.
# The Kafka timestamp is also kept for reference.
kafka_value_df = kafka_df.select(
    F.col("timestamp").alias("kafka_timestamp"),
    F.col("value").cast("string").alias("value")
)


# Deserialize the JSON string into Spark structured data.
# The producer sends one Kafka message as a list of accident records,
# so collision_batch_schema is used here.
parsed_batch_df = kafka_value_df.select(
    F.from_json(F.col("value"), collision_batch_schema).alias("records"),
    F.col("kafka_timestamp")
)

# Explode the array of records into individual rows.
# For example, if one Kafka message contains 80 accident records,
# this step converts it into 80 rows in the streaming DataFrame.
streaming_collision_df = (
    parsed_batch_df
    .select(
        F.explode(F.col("records")).alias("record"),
        F.col("kafka_timestamp")
    )
    .select(
        "record.*",
        "kafka_timestamp"
    )
    # Convert the numeric Unix timestamp from the producer
    # into a proper Spark timestamp column.
    .withColumn(
        "accident_ts_time",
        F.from_unixtime(F.col("accident_ts")).cast("timestamp")
    )
)

# Print the streaming DataFrame schema to confirm that
# the Kafka messages have been parsed into structured columns.
streaming_collision_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)



In [70]:
# Task 2.3 Convert streaming data into appropriate types

def try_cast_col(col_name, data_type):
    """
    Safely convert a string column into the required data type.
    Empty strings are converted to null, and invalid values become null
    instead of causing the streaming query to fail.
    """
    return F.expr(f"try_cast(nullif(trim(`{col_name}`), '') as {data_type})")


streaming_collision_typed_df = (
    streaming_collision_df
    .withColumn("longitude", try_cast_col("longitude", "double"))
    .withColumn("latitude", try_cast_col("latitude", "double"))
    .withColumn("speed_limit_num", try_cast_col("speed_limit", "double"))
    .withColumn("accident_ts_time", F.col("accident_ts_time").cast("timestamp"))
)

streaming_collision_typed_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)
 |-- speed_limit_num: double (nullable = true)



This step converts selected streaming columns into appropriate data types. Most Kafka message fields are initially parsed as strings, so numeric fields such as `longitude`, `latitude`, and `speed_limit` need to be converted before further processing. The `try_cast` logic is used to safely handle empty or invalid values by returning null instead of causing the streaming query to fail. The `accident_ts_time` column is also kept as a timestamp because it represents the simulated event time for streaming operations.

## 2.4 Event-Time Watermark

A 30-second watermark is applied to `accident_ts_time`. This tells Spark to keep event-time state for recent records while discarding records that arrive more than 30 seconds later than the current event-time watermark. In this local simulation, very late records may not appear, but the watermark is still required to make the streaming design consistent with the assignment requirement and to support windowed aggregations safely.

In [71]:
# Task 2.4: Apply watermark on event time

# Use accident_ts_time as event time and allow 30 seconds late data.
streaming_collision_watermark_df = (
    streaming_collision_typed_df
    .withWatermark("accident_ts_time", "30 seconds")
)

# Check schema after applying watermark.
streaming_collision_watermark_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)
 |-- speed_limit_num: double (nullable = true)



## 2.5 Recreating the A2A Feature Transformations

Before using the saved A2A model, the streaming data must be transformed into the same feature structure that the model expects. This section aggregates static vehicle information by `collision_index`, joins it to the streaming collision records, imputes missing vehicle-level numeric values, cleans categorical variables, and creates the engineered features used in A2A.

The important engineered features are:

- `Hour_clean`, derived from the accident time.
- `Peak_Traffic_clean`, where 07:00-09:00 and 16:00-18:00 are treated as peak traffic periods.
- `Bad_Condition_Risk`, which flags non-normal weather, road surface, or light conditions.
- Aggregated vehicle features such as number of vehicles, average driver age, vehicle age, engine capacity, and binary indicators for young drivers, older drivers, motorcycles, and large vehicles.

The final check confirms that all feature columns required by the saved pipeline are present before prediction.

In [72]:
# Task 2.5: Prepare static vehicle features

from pyspark.sql import functions as F


# Convert selected vehicle columns from string to numeric types.
vehicle_typed_df = (
    vehicle_df
    .withColumn("vehicle_reference_num", try_cast_col("vehicle_reference", "int"))
    .withColumn("age_of_driver_num", try_cast_col("age_of_driver", "double"))
    .withColumn("engine_capacity_cc_num", try_cast_col("engine_capacity_cc", "double"))
    .withColumn("age_of_vehicle_num", try_cast_col("age_of_vehicle", "double"))
)


# Aggregate vehicle records to create one summary row per collision.
vehicle_summary_df = (
    vehicle_typed_df
    .groupBy("collision_index")
    .agg(
        F.count("*").cast("double").alias("num_vehicles"),

        F.avg(
            F.when(F.col("age_of_driver_num") >= 0, F.col("age_of_driver_num"))
        ).alias("avg_age_of_driver"),

        F.avg(
            F.when(F.col("engine_capacity_cc_num") > 0, F.col("engine_capacity_cc_num"))
        ).alias("avg_engine_capacity_cc"),

        F.avg(
            F.when(F.col("age_of_vehicle_num") >= 0, F.col("age_of_vehicle_num"))
        ).alias("avg_age_of_vehicle"),

        F.first(
            F.when(F.col("vehicle_reference_num") == 1, F.col("vehicle_type")),
            ignorenulls=True
        ).alias("main_vehicle_type"),

        F.first(
            F.when(F.col("vehicle_reference_num") == 1, F.col("vehicle_manoeuvre")),
            ignorenulls=True
        ).alias("main_vehicle_manoeuvre"),

        F.first(
            F.when(F.col("vehicle_reference_num") == 1, F.col("sex_of_driver")),
            ignorenulls=True
        ).alias("main_sex_of_driver"),

        F.max(
            F.when(
                (F.col("age_of_driver_num") >= 17) & (F.col("age_of_driver_num") <= 25),
                1
            ).otherwise(0)
        ).cast("double").alias("has_young_driver"),

        F.max(
            F.when(F.col("age_of_driver_num") >= 65, 1).otherwise(0)
        ).cast("double").alias("has_old_driver"),

        F.max(
            F.when(F.col("vehicle_type").isin("2", "3", "4", "5"), 1).otherwise(0)
        ).cast("double").alias("has_motorcycle"),

        F.max(
            F.when(F.col("vehicle_type").isin("19", "20", "21", "98"), 1).otherwise(0)
        ).cast("double").alias("has_large_vehicle"),

        F.max(
            F.when(
                (F.col("engine_capacity_cc_num").isNull()) |
                (F.col("engine_capacity_cc_num") <= 0),
                1
            ).otherwise(0)
        ).cast("double").alias("engine_capacity_missing"),

        F.max(
            F.when(
                (F.col("age_of_vehicle_num").isNull()) |
                (F.col("age_of_vehicle_num") < 0),
                1
            ).otherwise(0)
        ).cast("double").alias("vehicle_age_missing")
    )
)

vehicle_summary_df.printSchema()
vehicle_summary_df.show(5, truncate=False)

root
 |-- collision_index: string (nullable = true)
 |-- num_vehicles: double (nullable = false)
 |-- avg_age_of_driver: double (nullable = true)
 |-- avg_engine_capacity_cc: double (nullable = true)
 |-- avg_age_of_vehicle: double (nullable = true)
 |-- main_vehicle_type: string (nullable = true)
 |-- main_vehicle_manoeuvre: string (nullable = true)
 |-- main_sex_of_driver: string (nullable = true)
 |-- has_young_driver: double (nullable = true)
 |-- has_old_driver: double (nullable = true)
 |-- has_motorcycle: double (nullable = true)
 |-- has_large_vehicle: double (nullable = true)
 |-- engine_capacity_missing: double (nullable = true)
 |-- vehicle_age_missing: double (nullable = true)

+---------------+------------+------------------+----------------------+------------------+-----------------+----------------------+------------------+----------------+--------------+--------------+-----------------+-----------------------+-------------------+
|collision_index|num_vehicles|avg_age_of

In [73]:
# Calculate simple imputation values from static vehicle summary

impute_row = vehicle_summary_df.agg(
    F.avg("avg_age_of_driver").alias("mean_age_driver"),
    F.avg("avg_engine_capacity_cc").alias("mean_engine_capacity"),
    F.avg("avg_age_of_vehicle").alias("mean_age_vehicle")
).first()

mean_age_driver = impute_row["mean_age_driver"]
mean_engine_capacity = impute_row["mean_engine_capacity"]
mean_age_vehicle = impute_row["mean_age_vehicle"]

print("mean_age_driver:", mean_age_driver)
print("mean_engine_capacity:", mean_engine_capacity)
print("mean_age_vehicle:", mean_age_vehicle)

mean_age_driver: 40.95286358648492
mean_engine_capacity: 2022.8038139969995
mean_age_vehicle: 8.581113344334726


In [74]:
# Join streaming collision data with static vehicle features

streaming_joined_df = (
    streaming_collision_watermark_df
    .join(vehicle_summary_df, on="collision_index", how="left")
)

streaming_joined_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)
 |-- speed_limit_num: double (nullable = true)
 |-- num_vehicles: double (nullable = true)
 |-- avg_age_of_driver: double (nullable 

In [75]:
# Feature engineering for streaming model input
from pyspark.ml import PipelineModel

# Load the trained A2A PipelineModel.
final_model = PipelineModel.load("final_gbt_model")

# Inspect categorical labels learned by the first pipeline stage.
indexer_stage = final_model.stages[0]

for col_name, labels in zip(indexer_stage.getInputCols(), indexer_stage.labelsArray):
    print("\nColumn:", col_name)
    print("Sample labels:", list(labels[:10]))


Column: road_type_clean
Sample labels: ['6', '3', '1', '12', 'Unknown', '2', '7']

Column: light_conditions_clean
Sample labels: ['1', '4', '6', '7', '5', 'Unknown']

Column: weather_conditions_clean
Sample labels: ['1', '2', '8', '4', '5', 'Unknown', '7', '3', '6']

Column: road_surface_conditions_clean
Sample labels: ['1', '2', '4', '3', '5', '6', '7']

Column: main_vehicle_type_clean
Sample labels: ['109', '9', '1', '104', '19', '11', '113', '2', '3', '5']

Column: main_vehicle_manoeuvre_clean
Sample labels: ['19', '9', '3', '4', '2', '7', '13', '14', '5', '10']

Column: Peak_Traffic_clean
Sample labels: ['Off-peak', 'Peak', 'Unknown']

Column: junction_detail_clean
Sample labels: ['0', '13', '16', '18', '19', '17', 'Unknown']

Column: pedestrian_crossing_clean
Sample labels: ['0', '14', '15', '13', 'Unknown', '17', '16', '11', '12']

Column: carriageway_hazards_clean
Sample labels: ['0', '13', '17', '20', '21', '11', 'Unknown', '18', '12', '14']

Column: main_sex_of_driver_clean
S

In [76]:
# Clean string columns before passing them into the A2A pipeline.
def clean_string_col(col_name):
    return (
        F.when(
            F.col(col_name).isNull() | (F.trim(F.col(col_name).cast("string")) == ""),
            F.lit("Unknown")
        )
        .otherwise(F.trim(F.col(col_name).cast("string")))
    )

# Create the same feature columns used during A2A model training.
streaming_model_input_df = (
    streaming_joined_df

    # Clean categorical columns from collision data
    .withColumn("road_type_clean", clean_string_col("road_type"))
    .withColumn("light_conditions_clean", clean_string_col("light_conditions"))
    .withColumn("weather_conditions_clean", clean_string_col("weather_conditions"))
    .withColumn("road_surface_conditions_clean", clean_string_col("road_surface_conditions"))
    .withColumn("junction_detail_clean", clean_string_col("junction_detail"))
    .withColumn("pedestrian_crossing_clean", clean_string_col("pedestrian_crossing"))
    .withColumn("carriageway_hazards_clean", clean_string_col("carriageway_hazards"))

    # Clean categorical columns from vehicle summary
    .withColumn("main_vehicle_type_clean", clean_string_col("main_vehicle_type"))
    .withColumn("main_vehicle_manoeuvre_clean", clean_string_col("main_vehicle_manoeuvre"))
    .withColumn("main_sex_of_driver_clean", clean_string_col("main_sex_of_driver"))

    # Area feature
    .withColumn("area_group_clean", clean_string_col("area"))

    # Extract hour from accident time.
    .withColumn(
        "Hour_clean",
        F.expr("try_cast(nullif(trim(split(`time`, ':')[0]), '') as double)")
    )
    
    .withColumn(
        "Hour_clean",
        F.when(F.col("Hour_clean").isNull(), F.lit(0.0))
         .otherwise(F.col("Hour_clean"))
    )

    # Create peak traffic category.
    .withColumn(
        "Peak_Traffic_clean",
        F.when(
            ((F.col("Hour_clean") >= 7) & (F.col("Hour_clean") <= 9)) |
            ((F.col("Hour_clean") >= 16) & (F.col("Hour_clean") <= 18)),
            F.lit("Peak")
        )
        .otherwise(F.lit("Off-peak"))
    )

    # Fill missing numeric features.
    .withColumn(
        "speed_limit_num",
        F.when(F.col("speed_limit_num").isNull(), F.lit(0.0))
         .otherwise(F.col("speed_limit_num"))
    )
    .withColumn(
        "num_vehicles",
        F.when(F.col("num_vehicles").isNull(), F.lit(1.0))
         .otherwise(F.col("num_vehicles"))
    )
    .withColumn(
        "avg_age_of_driver_imputed",
        F.when(F.col("avg_age_of_driver").isNull(), F.lit(mean_age_driver))
         .otherwise(F.col("avg_age_of_driver"))
    )
    .withColumn(
        "avg_engine_capacity_cc_imputed",
        F.when(F.col("avg_engine_capacity_cc").isNull(), F.lit(mean_engine_capacity))
         .otherwise(F.col("avg_engine_capacity_cc"))
    )
    .withColumn(
        "avg_age_of_vehicle_imputed",
        F.when(F.col("avg_age_of_vehicle").isNull(), F.lit(mean_age_vehicle))
         .otherwise(F.col("avg_age_of_vehicle"))
    )

    # Create road and environment risk feature.
    .withColumn(
        "Bad_Condition_Risk",
        F.when(
            (F.col("weather_conditions_clean") != "1") |
            (F.col("road_surface_conditions_clean") != "1") |
            (F.col("light_conditions_clean") != "1"),
            F.lit(1.0)
        )
        .otherwise(F.lit(0.0))
    )

    # Fill missing binary vehicle features after the left join.
    .withColumn("has_young_driver", F.coalesce(F.col("has_young_driver"), F.lit(0.0)))
    .withColumn("has_old_driver", F.coalesce(F.col("has_old_driver"), F.lit(0.0)))
    .withColumn("has_motorcycle", F.coalesce(F.col("has_motorcycle"), F.lit(0.0)))
    .withColumn("has_large_vehicle", F.coalesce(F.col("has_large_vehicle"), F.lit(0.0)))
    .withColumn("engine_capacity_missing", F.coalesce(F.col("engine_capacity_missing"), F.lit(1.0)))
    .withColumn("vehicle_age_missing", F.coalesce(F.col("vehicle_age_missing"), F.lit(1.0)))
)

streaming_model_input_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)
 |-- speed_limit_num: double (nullable = true)
 |-- num_vehicles: double (nullable = true)
 |-- avg_age_of_driver: double (nullable 

In [77]:
# Required input columns for the saved A2A PipelineModel.

categorical_input_cols = [
    "road_type_clean",
    "light_conditions_clean",
    "weather_conditions_clean",
    "road_surface_conditions_clean",
    "main_vehicle_type_clean",
    "main_vehicle_manoeuvre_clean",
    "Peak_Traffic_clean",
    "junction_detail_clean",
    "pedestrian_crossing_clean",
    "carriageway_hazards_clean",
    "main_sex_of_driver_clean",
    "area_group_clean"
]

numeric_input_cols = [
    "speed_limit_num",
    "num_vehicles",
    "avg_age_of_driver_imputed",
    "Hour_clean",
    "avg_engine_capacity_cc_imputed",
    "avg_age_of_vehicle_imputed",
    "Bad_Condition_Risk",
    "has_young_driver",
    "has_old_driver",
    "has_motorcycle",
    "has_large_vehicle",
    "engine_capacity_missing",
    "vehicle_age_missing"
]

# Check whether all required model input columns are available.
required_cols = categorical_input_cols + numeric_input_cols

missing_cols = []

for col_name in required_cols:
    if col_name not in streaming_model_input_df.columns:
        missing_cols.append(col_name)

print("Missing columns:", missing_cols)

Missing columns: []


This step prepares the streaming data so that it matches the input format used by the trained A2A PipelineModel. The saved model is loaded first, and the categorical labels learned during training are inspected to confirm the expected input structure.

The joined streaming data is then transformed into model-ready features. Categorical columns are cleaned by replacing null or empty values with `Unknown`, while accident time is used to derive `Hour_clean` and `Peak_Traffic_clean`. Numeric vehicle-related columns are imputed using mean values calculated from the static vehicle summary. Additional binary risk features are also prepared, including bad condition risk and vehicle-related indicators.

Finally, the code checks whether all categorical and numeric columns required by the A2A model are present in the streaming DataFrame. An empty `missing_cols` list means the streaming data has the required structure for real-time prediction.

## 2.6 Real-Time Prediction and Console Aggregations

The saved Spark ML `PipelineModel` from A2A is loaded and applied to the streaming feature dataframe. The model produces a continuous regression prediction, which is converted into a 1-to-10 severity class using the same threshold logic used after model training.

Three streaming outputs are then created according to the task requirement:

1. High-severity accidents with predicted severity greater than 7, printed every 5 seconds.
2. Accident counts by predicted severity class, updated every 10 seconds.
3. Low, medium, and high severity counts for each local district, updated every 30 seconds.

The intermediate memory-sink test is included to confirm that the model can transform streaming records successfully before the console and file outputs are started.

In [78]:
# Task 2.6: Load A2A PipelineModel and make predictions

from pyspark.ml import PipelineModel

# Load the trained regression pipeline from A2A.
final_model = PipelineModel.load("final_gbt_model")

# Apply the saved model to the prepared streaming input data.
prediction_df = final_model.transform(streaming_model_input_df)

prediction_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- urban_or_rural_area: string (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- accident_ts_time: timestamp (nullable = true)
 |-- speed_limit_num: double (nullable = true)
 |-- num_vehicles: double (nullable = true)
 |-- avg_age_of_driver: double (nullable 

### Severity Conversion Justification

The saved A2A model is a regression model, so the raw `prediction` column represents a continuous severity score rather than a direct severity class. Since Task 2.6 requires high-severity accidents to be identified using `predicted_severity > 7`, the continuous regression output is converted into severity levels from 1 to 10 using threshold-based severity bands.

This conversion is used as a post-processing step for streaming interpretation. The raw `prediction` value is retained as the original model output, while `predicted_severity` is used to support dashboard reporting and high-severity filtering.

The model tends to produce predictions within a compressed lower-to-medium range because high-severity accidents are rare in the historical data. This is a common issue in rare-event prediction tasks, where the model has fewer examples of extreme cases to learn from. Increasing the number of high-severity training examples, or using stratified sampling, could further improve the model’s ability to distinguish severe accidents. However, for this streaming task, the severity-band conversion provides a practical way to map the regression output back to the required 1 to 10 severity scale.


In [79]:
# Convert continuous regression prediction into severity bands from 1 to 10.

prediction_df = prediction_df.withColumn(
    "predicted_severity",
    F.when(F.col("prediction") < 2.04, 1)
     .when(F.col("prediction") < 2.64, 2)
     .when(F.col("prediction") < 3.43, 3)
     .when(F.col("prediction") < 4.05, 4)
     .when(F.col("prediction") < 4.74, 5)
     .when(F.col("prediction") < 5.22, 6)
     .when(F.col("prediction") < 5.39, 7)
     .when(F.col("prediction") < 5.54, 8)
     .when(F.col("prediction") < 5.73, 9)
     .otherwise(10)
     .cast("int")
)

### 2.6a: High-Severity Accident Output

This query filters the streaming prediction output to identify high-severity accidents, defined as records with `predicted_severity > 7`. The filtered records are written to the console every 5 seconds using a processing-time trigger. This satisfies the task requirement to print high-severity accident predictions from the streaming pipeline at regular 5-second intervals.

The output includes the collision identifier, event time, location, area, raw regression prediction, and converted predicted severity level.

In [80]:
# Task 2.6a: Print high-severity accidents every 5 seconds

import os
import shutil

# Stop any active streaming queries before starting a new one.
for q in spark.streams.active:
    q.stop()

checkpoint_path = "A2B/checkpoint/high_severity_console"

# Delete old checkpoint to avoid conflicts from previous runs.
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("Old checkpoint deleted.")
    

# Filter streaming predictions for high-severity accidents.
high_severity_df = (
    prediction_df
    .filter(F.col("predicted_severity") > 7)
    .select(
        "collision_index",
        "accident_ts_time",
        "longitude",
        "latitude",
        "area_group_clean",
        "prediction",
        "predicted_severity"
    )
)

# Print new high-severity accident records to the console every 5 seconds.
high_severity_query = (
    high_severity_df
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", checkpoint_path)
    .start()
)

Old checkpoint deleted.


This output shows high-severity accident records printed from the streaming prediction query every 5 seconds. The query filters records where `predicted_severity > 7`, so the displayed records represent accidents predicted as severity level 8, 9, or 10.

For each high-severity record, the output includes the collision identifier, event-time timestamp, longitude, latitude, local district, raw regression prediction, and converted predicted severity level. For example, if a record has `predicted_severity = 10`, it means the model output was mapped into the highest severity band for dashboard monitoring.

These records represent streamed collision records processed by the system, not necessarily real-world accidents occurring at exactly that rate.

In [17]:
# Stop 2.6a query
high_severity_query.stop()

### 2.6b: Accident Count by Predicted Severity

This query calculates the total number of predicted accidents for each severity level in 10-second event-time windows. The event time is based on `accident_ts_time`, which was created from the streaming timestamp sent by the Kafka producer.

The query groups the streaming prediction output by window and `predicted_severity`, then counts the number of accident records in each group. The result is written to the console every 10 seconds.

The `update` output mode is used because this task performs aggregation. As new records arrive within the same time window, the count for each predicted severity level can be updated. This is different from Task 2.6a, which uses `append` mode because it prints individual high-severity accident records rather than updating aggregated counts.


In [81]:
# Task 2.6b: Print total accidents for each severity every 10 seconds

import os
import shutil

# Stop any active streaming queries before starting a new one.
for q in spark.streams.active:
    q.stop()

checkpoint_path = "A2B/checkpoint/severity_count_console"

# Delete old checkpoint for a fresh demo run.
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("Old checkpoint deleted.")

# Count accidents for each predicted severity in 10-second windows.
severity_count_df = (
    prediction_df
    .groupBy(
        F.window(F.col("accident_ts_time"), "10 seconds"),
        F.col("predicted_severity")
    )
    .agg(
        F.count("*").alias("total_accidents")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("predicted_severity"),
        F.col("total_accidents")
    )
)

# Print updated severity counts to console every 10 seconds.
severity_count_query = (
    severity_count_df
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="10 seconds")
    .option("checkpointLocation", checkpoint_path)
    .start()
)

Old checkpoint deleted.


This output summarises the number of streamed collision records for each predicted severity level within 10-second event-time windows. For each window, the query groups records by `predicted_severity` and counts the total number of accident records in each severity level.

For example, in a 10-second window, a row with `predicted_severity = 10` and `total_accidents = 12` means that the system processed 12 collision records predicted as severity level 10 during that window. Similarly, other rows in the same time window show the number of records predicted as severity levels 1 to 9.

These counts represent streamed collision records processed by the system, not necessarily real-world accidents occurring at exactly that rate.

In [17]:
# Stop Task 2.6b query
severity_count_query.stop()

### 2.6c: District-Level Severity Counts

This query summarises predicted accident severity by local district every 30 seconds. The streaming prediction output is grouped using a 30-second event-time window based on `accident_ts_time` and the district feature represented by `area_group_clean`.

The predicted severity values are divided into three groups: low severity for levels 1 to 3, medium severity for levels 4 to 6, and high severity for levels 7 to 10. The query then counts how many accidents fall into each severity group for every district and prints the updated results to the console.

The `update` output mode is used because this task performs aggregation. As new streaming records arrive within the same window, the district-level counts can be updated until the window is complete.

In [82]:
# Task 2.6c: Count low, medium, and high severity accidents by district every 30 seconds

import os
import shutil

# Stop any active streaming queries before starting a new one.
for q in spark.streams.active:
    q.stop()

checkpoint_path = "A2B/checkpoint/district_severity_console"

# Delete old checkpoint for a fresh demo run.
if os.path.exists(checkpoint_path):
    shutil.rmtree(checkpoint_path)
    print("Old checkpoint deleted.")
else:
    print("Checkpoint folder does not exist.")

# Count low, medium, and high severity accidents by district in 30-second windows.
district_severity_df = (
    prediction_df
    .groupBy(
        F.window(F.col("accident_ts_time"), "30 seconds"),
        F.col("area_group_clean")
    )
    .agg(
        F.sum(
            F.when(
                (F.col("predicted_severity") >= 1) &
                (F.col("predicted_severity") <= 3),
                1
            ).otherwise(0)
        ).alias("low_severity_count"),

        F.sum(
            F.when(
                (F.col("predicted_severity") >= 4) &
                (F.col("predicted_severity") <= 6),
                1
            ).otherwise(0)
        ).alias("medium_severity_count"),

        F.sum(
            F.when(
                (F.col("predicted_severity") >= 7) &
                (F.col("predicted_severity") <= 10),
                1
            ).otherwise(0)
        ).alias("high_severity_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("area_group_clean").alias("local_district"),
        F.col("low_severity_count"),
        F.col("medium_severity_count"),
        F.col("high_severity_count")
    )
)

# Print updated district severity counts to the console every 30 seconds.
district_severity_query = (
    district_severity_df
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", "false")
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", checkpoint_path)
    .start()
)

Old checkpoint deleted.


This output summarises predicted accident severity by local district within 30-second event-time windows. For each district, the query counts how many streamed collision records are classified as low severity, medium severity, and high severity. For example, in the 02:15:00 to 02:15:30 window, the Metropolitan Police district had 319 low-severity records, 184 medium-severity records, and 12 high-severity records. These counts represent streamed collision records processed by the system, not necessarily real-world accidents occurring at exactly that rate.

In [19]:
# Stop task 2.6c query
district_severity_query.stop()

### 2.7: Save Streaming Outputs to Parquet

This step saves the three streaming outputs from Task 2.6 into Parquet folders. The first output stores high-severity accident records where `predicted_severity > 7`. The second output stores the total number of accidents for each predicted severity level. The third output stores district-level counts for low, medium, and high severity accidents.

Each output is written to a separate Parquet folder under `A2B/parquet/`, and each streaming query uses a separate checkpoint folder under `A2B/checkpoint/`. This allows Spark Structured Streaming to track the progress of each query independently.

In [83]:
# Task 2.7: Save Task 2.6 outputs to Parquet files

import shutil
import os

# Stop active streaming queries before cleaning output folders.
for q in spark.streams.active:
    q.stop()

# Prepare output paths and clean old folders
parquet_base_path = "A2B/parquet"
checkpoint_base_path = "A2B/checkpoint"

paths_to_clean = [
    f"{parquet_base_path}/high_severity",
    f"{parquet_base_path}/severity_count",
    f"{parquet_base_path}/district_severity",
    f"{checkpoint_base_path}/high_severity_parquet",
    f"{checkpoint_base_path}/severity_count_parquet",
    f"{checkpoint_base_path}/district_severity_parquet"
]

for path in paths_to_clean:
    if os.path.exists(path):
        shutil.rmtree(path)
        print("Deleted:", path)

# Use prediction output from the already-watermarked streaming pipeline.
prediction_output_df = prediction_df

Deleted: A2B/parquet/high_severity
Deleted: A2B/parquet/severity_count
Deleted: A2B/parquet/district_severity
Deleted: A2B/checkpoint/high_severity_parquet
Deleted: A2B/checkpoint/severity_count_parquet
Deleted: A2B/checkpoint/district_severity_parquet


In [84]:
# 7a. Output 1: High-severity accident records.
high_severity_parquet_df = (
    prediction_output_df
    .filter(F.col("predicted_severity") > 7)
    .select(
        "collision_index",
        "accident_ts_time",
        "longitude",
        "latitude",
        F.col("area_group_clean").alias("local_district"),
        "prediction",
        "predicted_severity"
    )
)

# Write high-severity records to Parquet.
high_severity_parquet_query = (
    high_severity_parquet_df
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", f"{parquet_base_path}/high_severity")
    .option("checkpointLocation", f"{checkpoint_base_path}/high_severity_parquet")
    .trigger(processingTime="5 seconds")
    .start()
)

The high-severity records are written in append mode because they are individual prediction records. The two aggregated outputs use `foreachBatch` because their counts are updated as new data arrives within each event-time window. The `batch_id` column is added to the aggregated Parquet outputs to show which micro-batch produced each saved result.

In [85]:
# 7b. Output 2: Total accidents for each predicted severity.

severity_count_parquet_df = (
    prediction_output_df
    .groupBy(
        F.window(F.col("accident_ts_time"), "10 seconds"),
        F.col("predicted_severity")
    )
    .agg(
        F.count("*").alias("total_accidents")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("predicted_severity"),
        F.col("total_accidents")
    )
)


# Write each updated micro-batch result to Parquet.
def write_severity_count_to_parquet(batch_df, batch_id):
    if batch_df.count() > 0:
        (
            batch_df
            .withColumn("batch_id", F.lit(batch_id))
            .write
            .mode("append")
            .parquet(f"{parquet_base_path}/severity_count")
        )


severity_count_parquet_query = (
    severity_count_parquet_df
    .writeStream
    .outputMode("update")
    .foreachBatch(write_severity_count_to_parquet)
    .option("checkpointLocation", f"{checkpoint_base_path}/severity_count_parquet")
    .trigger(processingTime="10 seconds")
    .start()
)

In [86]:
# 7c. Output 3: Low, medium, and high severity counts by district.

district_severity_parquet_df = (
    prediction_output_df
    .groupBy(
        F.window(F.col("accident_ts_time"), "30 seconds"),
        F.col("area_group_clean")
    )
    .agg(
        F.sum(
            F.when(
                (F.col("predicted_severity") >= 1) &
                (F.col("predicted_severity") <= 3),
                1
            ).otherwise(0)
        ).alias("low_severity_count"),

        F.sum(
            F.when(
                (F.col("predicted_severity") >= 4) &
                (F.col("predicted_severity") <= 6),
                1
            ).otherwise(0)
        ).alias("medium_severity_count"),

        F.sum(
            F.when(
                (F.col("predicted_severity") >= 7) &
                (F.col("predicted_severity") <= 10),
                1
            ).otherwise(0)
        ).alias("high_severity_count")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("area_group_clean").alias("local_district"),
        "low_severity_count",
        "medium_severity_count",
        "high_severity_count"
    )
)


# Write each updated micro-batch result to Parquet.
def write_district_severity_to_parquet(batch_df, batch_id):
    if batch_df.count() > 0:
        (
            batch_df
            .withColumn("batch_id", F.lit(batch_id))
            .write
            .mode("append")
            .parquet(f"{parquet_base_path}/district_severity")
        )


district_severity_parquet_query = (
    district_severity_parquet_df
    .writeStream
    .outputMode("update")
    .foreachBatch(write_district_severity_to_parquet)
    .option("checkpointLocation", f"{checkpoint_base_path}/district_severity_parquet")
    .trigger(processingTime="30 seconds")
    .start()
)

In [58]:
# Check the result and read Parquet outputs

import os
import glob

def read_parquet_if_ready(folder_path, table_name):
    """
    Read a Parquet folder only when actual parquet data files exist.
    """
    parquet_files = glob.glob(
        os.path.join(folder_path, "**", "*.parquet"),
        recursive=True
    )

    if len(parquet_files) > 0:
        print(f"{table_name} Parquet output is ready.")
        spark.read.parquet(folder_path).show(10, truncate=False)
    else:
        print(f"{table_name} Parquet folder exists, but data files are not ready yet.")


read_parquet_if_ready(
    "A2B/parquet/high_severity",
    "high_severity"
)

read_parquet_if_ready(
    "A2B/parquet/severity_count",
    "severity_count"
)

read_parquet_if_ready(
    "A2B/parquet/district_severity",
    "district_severity"
)

high_severity Parquet output is ready.
+---------------+-------------------+---------+---------+-------------------+------------------+------------------+
|collision_index|accident_ts_time   |longitude|latitude |local_district     |prediction        |predicted_severity|
+---------------+-------------------+---------+---------+-------------------+------------------+------------------+
|2025000000088  |2026-06-03 09:24:58|-1.484524|52.488064|Warwickshire       |6.06790150093848  |10                |
|2025000000118  |2026-06-03 09:24:58|-0.2288  |51.59981 |Metropolitan Police|5.896863228119151 |10                |
|2025000000135  |2026-06-03 09:24:58|0.75937  |52.200783|Suffolk            |5.8221413687714385|10                |
|2025000000244  |2026-06-03 09:25:00|-1.685972|54.916386|Northumbria        |5.912735790401928 |10                |
|2025000000339  |2026-06-03 09:25:01|-1.63073 |53.90462 |West Yorkshire     |5.508734915781405 |8                 |
|2025000000438  |2026-06-03 09:25

### 2.8: Stream Parquet Outputs to Kafka Topics

This step reads the three Parquet outputs created in Task 2.7 as streaming DataFrames and publishes them to separate Kafka topics. The high-severity accident records are sent to `a2b_high_severity`, the severity count summary is sent to `a2b_severity_count`, and the district-level severity summary is sent to `a2b_district_severity`.

The schemas are first read from the existing Parquet folders because Spark Structured Streaming requires a fixed schema when reading files as a stream. Each streaming DataFrame is then converted into JSON format using `to_json(struct(*))`, so the records can be written into the Kafka `value` column.

Separate checkpoint folders are used for each Kafka writing query. This allows Spark to track which Parquet files have already been processed and ensures that new messages are sent to Kafka when new Parquet batches appear.


In [87]:
# Task 2.8: Read Parquet files as streams and send them to Kafka topics

high_severity_topic = "a2b_high_severity"
severity_count_topic = "a2b_severity_count"
district_severity_topic = "a2b_district_severity"

hostip = "kafka"

# Read schemas from existing Parquet outputs
high_severity_schema = spark.read.parquet(
    "A2B/parquet/high_severity"
).schema

severity_count_schema = spark.read.parquet(
    "A2B/parquet/severity_count"
).schema

district_severity_schema = spark.read.parquet(
    "A2B/parquet/district_severity"
).schema

AnalysisException: Unable to infer schema for Parquet at . It must be specified manually.

In [60]:
# Stream 1: Read high-severity Parquet stream and send to Kafka

high_severity_parquet_stream = (
    spark.readStream
    .schema(high_severity_schema)
    .parquet("A2B/parquet/high_severity")
)

high_severity_kafka_df = high_severity_parquet_stream.select(
    F.to_json(F.struct("*")).alias("value")
)

high_severity_kafka_query = (
    high_severity_kafka_df
    .writeStream
    .outputMode("append")
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", high_severity_topic)
    .option("checkpointLocation", "A2B/checkpoint/high_severity_to_kafka")
    .trigger(processingTime="5 seconds")
    .start()
)


In [61]:
# Stream 2: Read severity-count Parquet stream and send to Kafka

severity_count_parquet_stream = (
    spark.readStream
    .schema(severity_count_schema)
    .parquet("A2B/parquet/severity_count")
)

severity_count_kafka_df = severity_count_parquet_stream.select(
    F.to_json(F.struct("*")).alias("value")
)

severity_count_kafka_query = (
    severity_count_kafka_df
    .writeStream
    .outputMode("append")
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", severity_count_topic)
    .option("checkpointLocation", "A2B/checkpoint/severity_count_to_kafka")
    .trigger(processingTime="10 seconds")
    .start()
)


In [62]:
# Stream 3: Read district-severity Parquet stream and send to Kafka

district_severity_parquet_stream = (
    spark.readStream
    .schema(district_severity_schema)
    .parquet("A2B/parquet/district_severity")
)

district_severity_kafka_df = district_severity_parquet_stream.select(
    F.to_json(F.struct("*")).alias("value")
)

district_severity_kafka_query = (
    district_severity_kafka_df
    .writeStream
    .outputMode("append")
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", district_severity_topic)
    .option("checkpointLocation", "A2B/checkpoint/district_severity_to_kafka")
    .trigger(processingTime="30 seconds")
    .start()
)


In [63]:
# Check Task 2.8 streaming queries

print("High severity to Kafka active:", high_severity_kafka_query.isActive)
print("Severity count to Kafka active:", severity_count_kafka_query.isActive)
print("District severity to Kafka active:", district_severity_kafka_query.isActive)

print("\nHigh severity progress:")
print(high_severity_kafka_query.lastProgress)

print("\nSeverity count progress:")
print(severity_count_kafka_query.lastProgress)

print("\nDistrict severity progress:")
print(district_severity_kafka_query.lastProgress)

High severity to Kafka active: True
Severity count to Kafka active: True
District severity to Kafka active: True

High severity progress:
{
    "id": "baf42ef2-328f-4b9c-890e-f031597cb8f5",
    "runId": "f9c2be51-48fc-4339-8935-6d82f5ecbb35",
    "name": null,
    "timestamp": "2026-06-12T05:59:55.001Z",
    "batchId": 306,
    "batchDuration": 188,
    "numInputRows": 8,
    "inputRowsPerSecond": 1.6,
    "processedRowsPerSecond": 42.6,
    "durationMs": {
        "addBatch": 47,
        "commitOffsets": 18,
        "getBatch": 7,
        "latestOffset": 82,
        "queryPlanning": 3,
        "triggerExecution": 188,
        "walCommit": 29
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "FileStreamSource[file:/home/student/A2B/A2B/parquet/high_severity]",
            "startOffset": {
                "logOffset": 305
            },
            "endOffset": {
                "logOffset": 306
            },
            "latestOffset": null,
      

The Task 2.8 streaming queries were successfully started and remained active. Each query reads one Parquet output from Task 2.7 as a streaming file source and writes the records to its corresponding Kafka topic.

The high-severity stream read 52,202 records from `A2B/parquet/high_severity` and wrote 52,202 messages to Kafka. The severity count stream read 22,930 records from `A2B/parquet/severity_count` and wrote 22,930 messages to Kafka. The district severity stream read 35,031 records from `A2B/parquet/district_severity` and wrote 35,031 messages to Kafka.

This confirms that the Parquet streaming outputs from Task 2.7 were successfully consumed and published to Kafka topics for use in the Part 3 dashboard.

In [89]:
# Stop all active streaming queries
for q in spark.streams.active:
    print("Stopping:", q.name, q.id)
    q.stop()

Stopping: None 6ba4d658-7848-4e89-921e-e37ea1c299ab
Stopping: None 7286f6c5-a9a9-4bc6-bb78-c30f380d8b71
Stopping: None f8c68ead-1032-4692-b65f-9ba722087e73


In [88]:
spark.streams.active

## Generative AI Usage Statement

Generative AI was used selectively in this assignment as a learning and support tool. The final code, testing decisions, interpretation of outputs, and notebook organisation were reviewed, adapted, and executed by me. I used Generative AI mainly to clarify concepts, improve code readability, debug errors, and draft explanatory markdown. I understand that I am responsible for the correctness, quality, and academic integrity of the submitted work.

### Task 2: Spark Structured Streaming and Prediction Pipeline

For Task 2, Generative AI was used to clarify Spark Structured Streaming concepts such as SparkSession configuration, checkpointing, schema definition, Kafka ingestion, JSON deserialisation, watermarking, static and streaming joins, feature preparation, model loading, prediction, windowed aggregation, writing streaming outputs to Parquet, and publishing Parquet streams back to Kafka. It was also used to help interpret console outputs, streaming progress logs, Parquet folder contents, and common errors such as reading Parquet files before data files were available. I adapted the suggested explanations and code structure to match my own A2A model pipeline and tested each streaming query in my notebook.
